# 🚢 Análise de Otimização de Frota - Accenture
## Pesquisa Operacional e Otimização Matemática

Este notebook apresenta uma análise completa de otimização de frota de navios usando técnicas de Pesquisa Operacional.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pulp
from IPython.display import display, Markdown

# Configurações
pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8')
sns.set_palette('husl')

## 1. Carregamento e Análise Inicial dos Dados

In [ ]:
# Carregar dados
df = pd.read_csv('../data/ships_data.csv')

display(Markdown(f"### 📊 Dataset: {len(df)} navios, {len(df.columns)} variáveis"))
display(df.head())

# Estatísticas básicas
display(Markdown("### 📈 Estatísticas Descritivas"))
display(df.describe())

## 2. Análise por Tipo de Navio

In [ ]:
analysis_by_type = df.groupby('ship_type').agg({
    'profit_usd': ['count', 'mean', 'sum', 'std'],
    'total_cost_usd': 'mean',
    'capacity_ton': 'mean',
    'fuel_efficiency': 'mean'
}).round(2)

display(Markdown("### 🚢 Análise de Performance por Tipo de Navio"))
display(analysis_by_type)

## 3. Modelo de Otimização com PuLP

In [ ]:
# Criar modelo de otimização
model = pulp.LpProblem("Otimizacao_Frota_Navios", pulp.LpMaximize)

# Variáveis de decisão
ship_vars = pulp.LpVariable.dicts("SelecionarNavio", df.index, cat=pulp.LpBinary)

# Função objetivo: Maximizar lucro total
model += pulp.lpSum([df.loc[i, 'profit_usd'] * ship_vars[i] for i in df.index])

# Restrições
# 1. Capacidade máxima da frota (80%)
model += pulp.lpSum([ship_vars[i] for i in df.index]) <= len(df) * 0.8

# 2. Navios em manutenção não podem operar
for i in df[df['maintenance_due']].index:
    model += ship_vars[i] == 0

display(Markdown("### ⚡ Modelo de Otimização Criado"))
print(f"Variáveis de decisão: {len(ship_vars)}")
print(f"Restrições: {len(model.constraints)}")

In [ ]:
# Resolver o modelo
model.solve()

display(Markdown("### 📊 Resultados da Otimização"))
print(f"Status: {pulp.LpStatus[model.status]}")
print(f"Lucro Total Otimizado: USD {pulp.value(model.objective):,.2f}")

# Extrair solução
selected_ships = [i for i in df.index if pulp.value(ship_vars[i]) == 1]
solution_df = df.loc[selected_ships].copy()

print(f"Navios selecionados: {len(selected_ships)}/{len(df)}")
print(f"Taxa de utilização: {len(selected_ships)/len(df)*100:.1f}%")

## 4. Visualização dos Resultados

In [ ]:
# Comparação antes/depois
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# 1. Distribuição de lucros
axes[0,0].hist(df['profit_usd'], bins=30, alpha=0.5, label='Todos Navios', color='blue')
axes[0,0].hist(solution_df['profit_usd'], bins=30, alpha=0.7, label='Selecionados', color='green')
axes[0,0].set_xlabel('Lucro (USD)')
axes[0,0].set_ylabel('Frequência')
axes[0,0].set_title('Distribuição de Lucros: Antes vs Depois')
axes[0,0].legend()
axes[0,0].grid(True, alpha=0.3)

# 2. Composição da frota selecionada
solution_df['ship_type'].value_counts().plot(kind='pie', ax=axes[0,1], autopct='%1.1f%%')
axes[0,1].set_title('Composição da Frota Selecionada')

# 3. Métricas de comparação
metrics = ['Lucro Total', 'Lucro Médio', 'Navios']
before = [df['profit_usd'].sum(), df['profit_usd'].mean(), len(df)]
after = [solution_df['profit_usd'].sum(), solution_df['profit_usd'].mean(), len(solution_df)]

x = np.arange(len(metrics))
width = 0.35
axes[1,0].bar(x - width/2, before, width, label='Antes', alpha=0.7)
axes[1,0].bar(x + width/2, after, width, label='Depois', alpha=0.7)
axes[1,0].set_xlabel('Métricas')
axes[1,0].set_ylabel('Valores')
axes[1,0].set_title('Comparação: Antes vs Depois da Otimização')
axes[1,0].set_xticks(x)
axes[1,0].set_xticklabels(metrics)
axes[1,0].legend()
axes[1,0].grid(True, alpha=0.3)

# 4. Eficiência dos navios selecionados
axes[1,1].scatter(solution_df['fuel_efficiency'], solution_df['profit_usd'], 
                 alpha=0.6, c=solution_df['capacity_ton'], cmap='viridis')
axes[1,1].set_xlabel('Eficiência de Combustível')
axes[1,1].set_ylabel('Lucro (USD)')
axes[1,1].set_title('Relação: Eficiência vs Lucro (Navios Selecionados)')
plt.colorbar(axes[1,1].collections[0], ax=axes[1,1], label='Capacidade (ton)')
axes[1,1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Insights e Recomendações

In [ ]:
display(Markdown("### 💡 PRINCIPAIS INSIGHTS"))

print(f"🎯 EFICIÊNCIA DA OTIMIZAÇÃO:")
print(f"   • Lucro total aumentou em: {(solution_df['profit_usd'].sum()/df['profit_usd'].sum()*100 - 100):.1f}%")
print(f"   • Frota otimizada: {len(solution_df)} navios ({len(solution_df)/len(df)*100:.1f}% da frota total)")
print(f"   • Lucro médio por navio: USD {solution_df['profit_usd'].mean():,.2f}")

print(f"\n🚢 COMPOSIÇÃO IDEAL:")
for ship_type in solution_df['ship_type'].value_counts().items():
    percentage = (ship_type[1] / len(solution_df)) * 100
    print(f"   • {ship_type[0]}: {ship_type[1]} navios ({percentage:.1f}%)")

print(f"\n📈 MÉTRICAS DE PERFORMANCE:")
print(f"   • Eficiência média: {solution_df['fuel_efficiency'].mean():.2f}")
print(f"   • Utilização média: {solution_df['utilization_rate'].mean():.1%}")
print(f"   • Velocidade média: {solution_df['speed_knots'].mean():.1f} nós")